<a href="https://colab.research.google.com/github/djamil13/Postdoc-Medical-AI-Cchallenge/blob/main/task2_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
TASK 2: Medical Report Generation with VLM
# task2_report_generation/
# ├── generate_reports.py
# ├── prompt_experiments.py
# ├── analyze_reports.py
# └── compare_with_cnn.py
Step 1: Report Generation with MedGemma
from transformers import AutoProcessor, AutoModelForPreTraining
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
class MedicalReportGenerator:
    def __init__(self, model_name="google/med-gemma-2b"):
        print(f"Loading model: {model_name}")
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = AutoModelForPreTraining.from_pretrained(model_name)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        print(f"Model loaded on {self.device}")
        def preprocess_image(self, image_np):
        """Convert numpy array to PIL image and resize for model"""
        if image_np.shape[0] == 1:  # channels first
            image_np = image_np[0]
                # Scale to 0-255 and convert to uint8
        image_np = (image_np * 255).astype(np.uint8)
                # Convert to PIL and resize to model expected size
        pil_image = Image.fromarray(image_np, mode='L').convert('RGB')
        return pil_image
        def generate_report(self, image, prompt_type="standard"):
        """Generate medical report with different prompting strategies"""
                pil_image = self.preprocess_image(image)
                # Different prompting strategies
        prompts = {
            "standard": "Describe this chest X-ray:",
            "detailed": "Provide a detailed radiology report for this chest X-ray including findings and impression:",
            "clinical": "As a radiologist, analyze this chest X-ray. Describe any abnormalities, location, severity, and provide an impression:",
            "structured": "Generate a structured radiology report with sections: FINDINGS, IMPRESSION, and RECOMMENDATION:",
            "comparative": "Compare this chest X-ray to a normal study. Describe any findings suggestive of pneumonia:"
        }
                prompt = prompts.get(prompt_type, prompts["standard"])
                # Process inputs
        inputs = self.processor(text=prompt, images=pil_image, return_tensors="pt").to(self.device)
                # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.7,
                do_sample=True,
                top_p=0.95
            )
                report = self.processor.decode(outputs[0], skip_special_tokens=True)
        # Remove the prompt from the output
        report = report.replace(prompt, "").strip()
                return report, prompt_type
        def experiment_prompts(self, image, image_id, true_label):
        """Try all prompting strategies and compare"""
                results = {}
        for prompt_type in ["standard", "detailed", "clinical", "structured", "comparative"]:
            report, _ = self.generate_report(image, prompt_type)
            results[prompt_type] = report
                return results
Step 2: Systematic Prompt Experimentation
python
# prompt_experiments.py
def run_prompt_experiments(generator, test_images, test_labels, num_samples=10):
    """Test different prompting strategies systematically"""
        all_results = []
        for idx in range(min(num_samples, len(test_images))):
        image = test_images[idx]
        true_label = test_labels[idx]

        print(f"\n{'='*60}")
        print(f"Sample {idx+1} - True Label: {'Pneumonia' if true_label == 1 else 'Normal'}")
        print(f"{'='*60}")
                # Try all prompts
        results = generator.experiment_prompts(image, idx, true_label)
                for prompt_type, report in results.items():
            print(f"\n🔹 PROMPT: {prompt_type}")
            print(f"REPORT: {report[:200]}...")

            all_results.append({
                'image_id': idx,
                'true_label': true_label,
                'prompt_type': prompt_type,
                'report': report
            })
                # Visualize image
        plt.figure(figsize=(4, 4))
        img_display = image[0] if image.shape[0] == 1 else image
        plt.imshow(img_display, cmap='gray')
        plt.title(f"Image {idx+1} - {'Pneumonia' if true_label == 1 else 'Normal'}")
        plt.axis('off')
        plt.savefig(f'sample_image_{idx}.png')
        plt.show()
        return all_results
Step 3: Compare VLM with CNN (CRITICAL INSIGHT)
python
# compare_with_cnn.py
def compare_vlm_with_cnn(generator, cnn_results, test_images, test_labels, num_samples=20):
    """Compare VLM reports with CNN predictions - KEY INSIGHT"""
        cnn_preds = cnn_results['predictions']
    cnn_probs = cnn_results['probabilities']
        # Select images from different categories
    categories = {
        'correct_pneumonia': [],  # CNN correct on pneumonia
        'correct_normal': [],      # CNN correct on normal
        'false_negative': [],      # CNN missed pneumonia (CRITICAL)
        'false_positive': []       # CNN false alarm
    }
        for i in range(len(test_labels)):
        if len(categories['correct_pneumonia']) < 5 and test_labels[i] == 1 and cnn_preds[i] == 1:
            categories['correct_pneumonia'].append(i)
        elif len(categories['correct_normal']) < 5 and test_labels[i] == 0 and cnn_preds[i] == 0:
            categories['correct_normal'].append(i)
        elif len(categories['false_negative']) < 5 and test_labels[i] == 1 and cnn_preds[i] == 0:
            categories['false_negative'].append(i)
        elif len(categories['false_positive']) < 5 and test_labels[i] == 0 and cnn_preds[i] == 1:
            categories['false_positive'].append(i)
        # Generate reports for all categories
    comparison_results = []
        for category, indices in categories.items():
        print(f"\n{'='*60}")
        print(f"CATEGORY: {category.upper()}")
        print(f"{'='*60}")
                for idx in indices:
            image = test_images[idx]
            true_label = test_labels[idx]
            cnn_pred = cnn_preds[idx]
            cnn_prob = cnn_probs[idx]
                        # Generate report with clinical prompt
            report, _ = generator.generate_report(image, prompt_type="clinical")
                        comparison_results.append({
                'category': category,
                'image_id': idx,
                'true_label': true_label,
                'cnn_prediction': cnn_pred,
                'cnn_confidence': cnn_prob,
                'vlm_report': report
            })
                        print(f"\nImage {idx}:")
            print(f"True: {'Pneumonia' if true_label == 1 else 'Normal'}")
            print(f"CNN: {'Pneumonia' if cnn_pred == 1 else 'Normal'} (conf: {cnn_prob:.2f})")
            print(f"VLM Report: {report[:150]}...")
        return comparison_results
Step 4: Analyze VLM Reports for Clinical Accuracy
def analyze_vlm_reports(comparison_results):
    """Analyze VLM reports for clinical accuracy and hallucinations"""
        analysis = {
        'false_negative_analysis': [],
        'false_positive_analysis': [],
        'hallucination_count': 0,
        'clinical_alignment': []
    }
        for result in comparison_results:
        category = result['category']
        report = result['vlm_report'].lower()
        true_label = result['true_label']

        # Check for key clinical terms
        pneumonia_terms = ['opacity', 'infiltrate', 'consolidation', 'pneumonia', 'air space']
        normal_terms = ['clear', 'normal', 'unremarkable', 'no findings']
                found_pneumonia_terms = [term for term in pneumonia_terms if term in report]
        found_normal_terms = [term for term in normal_terms if term in report]
                # Check for hallucinations (claiming findings not present)
        if true_label == 0 and len(found_pneumonia_terms) > 2:
            analysis['hallucination_count'] += 1
            analysis['false_positive_analysis'].append({
                'image_id': result['image_id'],
                'report': result['vlm_report'],
                'hallucinated_terms': found_pneumonia_terms
            })
                # Check for missed findings
        if true_label == 1 and len(found_pneumonia_terms) == 0:
            analysis['false_negative_analysis'].append({
                'image_id': result['image_id'],
                'report': result['vlm_report']
            })
                # Clinical alignment with CNN
        if category == 'false_negative':
            # CNN missed, what does VLM say?
            if len(found_pneumonia_terms) > 0:
                analysis['clinical_alignment'].append({
                    'image_id': result['image_id'],
                    'insight': f"VLM detected findings that CNN missed: {found_pneumonia_terms}"
                })
        # Print analysis
    print("\n" + "="*60)
    print("🔬 VLM REPORT ANALYSIS")
    print("="*60)
    print(f"Total hallucinations (findings in normal images): {analysis['hallucination_count']}")
    print(f"False negative cases where VLM detected findings: {len(analysis['clinical_alignment'])}")
        if analysis['clinical_alignment']:
        print("\n💡 KEY INSIGHT:")
        print("For images misclassified by CNN, VLM still identifies pneumonia-related findings.")
        print("This suggests VLM could serve as a complementary tool to catch missed cases.")
        for insight in analysis['clinical_alignment']:
            print(f"  - {insight['insight']}")
        return analysis
